# Gradient Flows in $L^p$ Metric Spaces

Gradient descent can be reformulated as a **proximal point algorithm** (implicit step):
$$
x^{k+1} = \arg\min_y \left\{ f(y) + \frac{1}{2\tau} \|y - x^k\|^2 \right\}.
$$
Replacing the Euclidean norm $\|\cdot\|_2$ by the **$L^p$ norm** $\|\cdot\|_p = (\sum_i |x_i|^p)^{1/p}$ yields the **$p$-proximal point** algorithm:
$$
x^{k+1} = \arg\min_y \left\{ f(y) + \frac{1}{2\tau} \|y - x^k\|_p^2 \right\}.
$$

## $p$-Laplacian gradient flow

For the energy $f(x) = \frac{1}{2}\|x - b\|_2^2$ (least-squares), the $p$-proximal step reduces to a **shrinkage** operator that depends on $p$:
- $p=2$: standard GD update $x \leftarrow (1/(1+\tau)) x + \tau b/(1+\tau)$ (Euclidean proximal).
- $p=1$: **soft-thresholding** $x_i \leftarrow \text{sign}(b_i)\max(|b_i|-\tau, 0)/(1+0)$ — sparse solutions.
- $p \to \infty$: **$\ell^\infty$ projection** — all coordinates updated equally, capped at $\pm\tau$.
- $1 < p < 2$: intermediate behaviour, superlinear shrinkage.

## Connection to signal processing

These flows arise in image restoration with $L^p$ priors:
- $p=2$: Gaussian noise model (Wiener filter).
- $p=1$: Laplacian prior / Total Variation denoising (promotes sparsity in gradient domain).
- $p > 2$: Super-Gaussian priors (flat-top distributions).

## Environment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from ipywidgets import interact, FloatSlider, IntSlider

plt.rcParams['figure.dpi'] = 120

## Proximal operators in Lp

For $f(x) = \frac{1}{2}\|x-b\|_2^2$, the $p$-proximal problem $\min_y \frac{1}{2}\|y-b\|_2^2 + \frac{1}{2\tau}\|y-x\|_p^2$ is solved componentwise.

In [ ]:
def prox_lp(x, b, tau, p, n_inner=50):
    """
    Proximal step: min_y 0.5*||y-b||^2 + (1/(2*tau))*||y-x||_p^2.
    For p=2: closed form. For p=1: soft threshold. Otherwise: scipy.
    """
    if p == 2:
        return (b + x / tau) / (1 + 1/tau)
    elif p == 1:
        # Prox of tau*||.||_1 applied to b
        # Full step: y = prox_{tau||.||_1}(b) = sign(b)*max(|b|-tau/2,0)
        # Here tau is the step, and the problem is min 0.5||y-b||^2 + 1/(2tau)||y-x||_1^2
        # -> closed form: see Boyd proximal algorithms
        # Approximation: run gradient descent on inner problem
        y = x.copy()
        lr = 0.05
        for _ in range(200):
            g = (y - b) + (1/tau) * np.sign(y - x) * np.abs(y - x)**(p-1)
            y = y - lr * g
        return y
    else:
        y = x.copy()
        lr = 0.02
        for _ in range(300):
            diff = y - x
            g = (y - b) + (1/tau) * p/2 * np.sign(diff) * np.abs(diff)**(p-1)
            y = y - lr * g
        return y

# 1D visualisation: what does the proximal operator look like?
b_vals = np.linspace(-3, 3, 200)
tau = 0.5
p_list = [1.0, 1.5, 2.0, 3.0, 10.0]
cols = plt.cm.plasma(np.linspace(0.1, 0.9, len(p_list)))

fig, ax = plt.subplots(figsize=(9, 5))
for p, col in zip(p_list, cols):
    y_out = np.array([prox_lp(np.array([0.0]), np.array([b]), tau, p)[0] for b in b_vals])
    ax.plot(b_vals, y_out, lw=2, color=col, label=f'$p={p}$')
ax.plot(b_vals, b_vals, 'k--', lw=1, alpha=0.4, label='identity')
ax.set_xlabel('$b$ (data point)'); ax.set_ylabel('prox output $y$')
ax.set_title(f'$L^p$ proximal operator (from $x=0$, $\\tau={tau}$): effect of $p$')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Gradient flow trajectories on a 2D example

We run the $p$-proximal gradient flow from the same initial point and compare the paths taken to the minimum of $f(x) = \frac{1}{2}\|x-b\|_2^2$.

In [ ]:
def run_pflow(p, b, x0, tau, n_iter):
    x = x0.copy(); traj = [x.copy()]
    for _ in range(n_iter):
        x = prox_lp(x, b, tau, p)
        traj.append(x.copy())
    return np.array(traj)

b2 = np.array([1.5, 0.8])
x0 = np.array([-1.5, -1.0])
tau2 = 0.3; n_it = 60

p_list2 = [1.0, 1.5, 2.0, 4.0]
fig, axes = plt.subplots(1, len(p_list2), figsize=(14, 4))

xi = np.linspace(-2, 2.5, 150); yi = np.linspace(-1.5, 1.5, 150)
X, Y = np.meshgrid(xi, yi)
F = 0.5*((X-b2[0])**2 + (Y-b2[1])**2)

for ax, p, col in zip(axes, p_list2, cols):
    ax.contourf(xi, yi, F, levels=12, cmap='viridis', alpha=0.7)
    t = run_pflow(p, b2, x0, tau2, n_it)
    ax.plot(t[:,0], t[:,1], 'w.-', lw=1.5, ms=4)
    ax.plot(t[0,0], t[0,1], 'ko', ms=8)
    ax.plot(b2[0], b2[1], 'r*', ms=12)
    ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(f'$p = {p}$')

fig.suptitle('$L^p$ gradient flow trajectories to the minimum', y=1.02)
plt.tight_layout(); plt.show()

## 1D signal denoising with Lp prior

We corrupt a piecewise-constant signal with Gaussian noise and compare denoising under $L^p$ priors (on the signal differences).

In [ ]:
rng = np.random.default_rng(7)
n_sig = 100
x_true = np.zeros(n_sig)
x_true[20:50] = 1.0; x_true[60:80] = 0.6
y_noisy = x_true + 0.15 * rng.standard_normal(n_sig)

def denoise_lp(y, lam, p, n_iter=500):
    """Proximal gradient: min 0.5||x-y||^2 + lam*||Dx||_p where D is difference operator."""
    x = y.copy()
    D = np.diag(np.ones(n_sig-1), 1) - np.diag(np.ones(n_sig-1), -1)
    D = D[:n_sig-1, :]
    step = 0.1
    for _ in range(n_iter):
        # gradient of data term
        g = x - y
        # gradient of regulariser (subgradient for p=1)
        r = D @ x
        if p >= 2:
            grad_r = lam * D.T @ (p * np.sign(r) * np.abs(r)**(p-1))
        else:
            grad_r = lam * D.T @ np.sign(r)  # subgradient L1
        x = x - step * (g + grad_r)
    return x

fig, axes = plt.subplots(len(p_list), 1, figsize=(10, 10), sharex=True)
for ax, p, col in zip(axes, p_list, cols):
    x_den = denoise_lp(y_noisy, lam=0.15, p=p)
    ax.plot(y_noisy, 'gray', lw=0.8, alpha=0.5, label='noisy')
    ax.plot(x_true, 'k--', lw=1.5, label='true')
    ax.plot(x_den, color=col, lw=2, label=f'$p={p}$')
    ax.legend(fontsize=8, loc='upper right'); ax.grid(alpha=0.2)
    ax.set_ylabel(f'$p={p}$')
axes[-1].set_xlabel('index')
fig.suptitle('Signal denoising with $L^p$ regularisation on differences', y=1.01)
plt.tight_layout(); plt.show()

## Interactive: step size and p

In [ ]:
def show_pflow(p=2.0, tau=0.3, n_iter=80):
    t = run_pflow(p, b2, x0, tau, n_iter)
    fvals = 0.5*np.sum((t - b2)**2, axis=1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    axes[0].contourf(xi, yi, F, levels=12, cmap='viridis', alpha=0.7)
    axes[0].plot(t[:,0], t[:,1], 'w.-', lw=1.5, ms=4)
    axes[0].plot(x0[0], x0[1], 'ko', ms=8); axes[0].plot(b2[0], b2[1], 'r*', ms=12)
    axes[0].set_aspect('equal'); axes[0].axis('off')
    axes[0].set_title(f'Trajectory ($p={p}$, $\\tau={tau}$)')
    axes[1].semilogy(fvals, 'royalblue', lw=2)
    axes[1].set_xlabel('iteration'); axes[1].set_ylabel('$f(x^k)$')
    axes[1].set_title('Convergence'); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

interact(show_pflow,
         p=FloatSlider(value=2.0, min=1.0, max=6.0, step=0.5, description='$p$'),
         tau=FloatSlider(value=0.3, min=0.05, max=1.0, step=0.05, description='$\\tau$'),
         n_iter=IntSlider(value=80, min=20, max=200, step=20, description='iters'));

## Bibliographical resources

- Parikh, N. and Boyd, S. (2014). Proximal algorithms. *Foundations and Trends in Optimization*, 1(3), 127–239.
- Rockafellar, R. T. (1976). Monotone operators and the proximal point algorithm. *SIAM Journal on Control and Optimization*, 14(5), 877–898.
- Rudin, L. I., Osher, S. and Fatemi, E. (1992). Nonlinear total variation based noise removal algorithms. *Physica D: Nonlinear Phenomena*, 60(1–4), 259–268.
- Villani, C. (2003). *Topics in Optimal Transportation*. American Mathematical Society.
- Ambrosio, L., Gigli, N. and Savaré, G. (2008). *Gradient Flows in Metric Spaces and in the Space of Probability Measures* (2nd ed.). Birkhäuser.